# Student Stress, Sleep & Screen Time Analysis

An exploratory data analysis examining factors that may be associated with student stress, with a focus on sleep, screen time, study hours, caffeine intake, age, and physical activity.

**Research question:** What factors appear to be associated with student stress?

### Hypotheses
1. Students who sleep less will have higher stress.
2. More screen time will be associated with higher stress.
3. A greater amount of exercise will reduce stress.

> **Note:** This notebook reproduces and organizes the exploratory analysis documented in the accompanying report, and extends it with a direct test of each hypothesis against `stress_level`, plus a set of follow-up questions raised during the initial exploration. It does not claim to establish causal relationships.

## 1. Setup

This notebook uses **Python, pandas, NumPy, SciPy, and Matplotlib** for data loading, descriptive analysis, statistical comparison, and visualization.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# Place the CSV file in the same directory as this notebook.
df = pd.read_csv('student_stress_sleep_screen.csv')

print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

## 2. Dataset Overview

Each row represents data for a student with a particular student ID. The variables documented in the original analysis are:

- `student_id`
- `age`
- `gender`
- `sleep_hours`
- `screen_time_hours`
- `stress_level`
- `study_hours`
- `physical_activity`
- `caffeine_intake`
- `academic_pressure`

`stress_level` (Low / Medium / High) is treated as the target variable in the original project.

In [ ]:
# Inspect the columns and data types.
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('dtype'))

print('\nMissing values:')
display(df.isna().sum().to_frame('missing_values'))

## 3. Numerical Variable Exploration

The original analysis examined the distributions of age, sleep hours, screen time, study hours, and caffeine intake. The summary below provides the same descriptive-statistics framework in a reproducible form.

In [ ]:
numeric_columns = [
    'age',
    'sleep_hours',
    'screen_time_hours',
    'study_hours',
    'caffeine_intake'
]

summary = df[numeric_columns].describe().T
summary = summary.rename(columns={
    '25%': 'Q1',
    '50%': 'Median',
    '75%': 'Q3'
})
summary = summary[['count', 'mean', 'std', 'min', 'Q1', 'Median', 'Q3', 'max']]
summary.round(2)

### Reference values from the original analysis

The original report recorded the following descriptive statistics, which can be used as a quick check that the correct dataset has been loaded:

| Variable | Mean | SD | Min | Median | Max |
|---|---:|---:|---:|---:|---:|
| Age | 21.53 | 2.24 | 18.00 | 22.00 | 25.00 |
| Sleep hours | 6.55 | 1.44 | 4.00 | 6.50 | 9.00 |
| Screen time hours | 7.11 | 2.91 | 2.00 | 7.30 | 12.00 |
| Study hours | 5.02 | 1.76 | 2.00 | 5.00 | 8.00 |
| Caffeine intake | 1.98 | 1.39 | 0.00 | 2.00 | 4.00 |

These values come from the original project report.

## 4. Visualizing Numerical Variables

The original project included five histograms: sleep hours, screen time, age, study hours, and caffeine intake.

In [ ]:
def plot_histogram(column, title, xlabel, bins=20):
    plt.figure(figsize=(8, 5))
    plt.hist(df[column].dropna(), bins=bins, edgecolor='white')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Number of Students')
    plt.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_histogram(
    'sleep_hours',
    'Distribution of Sleep Hours Among Students',
    'Sleep Hours'
)

### Figure 1 - Sleep Hours

The original analysis described the distribution as slightly left-skewed and noted that it appeared relatively uniform.

In [ ]:
plot_histogram(
    'screen_time_hours',
    'Distribution of Screen Time Hours Among Students',
    'Screen Time Hours'
)

### Figure 2 - Screen Time

The original analysis described the distribution as slightly right-skewed but notably uniform.

In [ ]:
plot_histogram(
    'age',
    'Distribution of Age Among Students',
    'Age'
)

### Figure 3 - Age

The original analysis noted a noticeable concentration of observations around ages 24-25.

In [ ]:
plot_histogram(
    'study_hours',
    'Distribution of Study Hours Among Students',
    'Study Hours'
)

### Figure 4 - Study Hours

The original analysis described the distribution as remarkably uniform with a slight right skew.

In [ ]:
plot_histogram(
    'caffeine_intake',
    'Distribution of Caffeine Intake Among Students',
    'Caffeine Intake'
)

### Figure 5 - Caffeine Intake

The original analysis described the distribution as slightly right-skewed, with an average caffeine-intake value of approximately 2.

## 5. Relationship Analysis

The original report examined two relationships in detail:

1. Screen time vs. sleep hours
2. Caffeine intake vs. sleep hours

The original expectations were negative relationships in both cases.

In [ ]:
def plot_scatter(x, y, title, xlabel, ylabel):
    plt.figure(figsize=(8, 6))
    plt.scatter(
        df[x],
        df[y],
        s=30,
        alpha=0.75,
        edgecolors='white'
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

plot_scatter(
    'screen_time_hours',
    'sleep_hours',
    'Hours of Screen Time vs. Hours of Sleep',
    'Hours of Screen Time',
    'Hours of Sleep'
)

### Figure 6 - Screen Time vs. Sleep

The original analysis expected a negative relationship but observed no obvious trend. It also noted the unusually even density of the points and questioned whether the dataset might have been synthetically generated. This is treated as a **data-quality concern/hypothesis**, not a definitive conclusion.

In [ ]:
screen_sleep_corr = df['screen_time_hours'].corr(df['sleep_hours'])
print(f'Pearson correlation (screen time vs. sleep): {screen_sleep_corr:.3f}')

In [ ]:
plot_scatter(
    'caffeine_intake',
    'sleep_hours',
    'Caffeine Intake vs. Hours of Sleep',
    'Caffeine Intake',
    'Hours of Sleep'
)

### Figure 7 - Caffeine Intake vs. Sleep

The original analysis expected a negative relationship but observed no apparent trend. The unusually even distribution was noted as another reason to question the realism or provenance of the dataset.

In [ ]:
caffeine_sleep_corr = df['caffeine_intake'].corr(df['sleep_hours'])
print(f'Pearson correlation (caffeine intake vs. sleep): {caffeine_sleep_corr:.3f}')

## 6. Testing the Three Hypotheses Against Stress

The relationships above (screen time vs. sleep, caffeine vs. sleep) turned out to be flat. The more direct test is each hypothesis against the actual target variable, `stress_level`. `stress_level` is ordinal (Low / Medium / High), so group means and **Spearman rank correlation** are used instead of Pearson correlation.

In [ ]:
stress_order = ['Low', 'Medium', 'High']
df['stress_level'] = pd.Categorical(df['stress_level'], categories=stress_order, ordered=True)
df['stress_rank'] = df['stress_level'].cat.codes

### Hypothesis 1: Students who sleep less will have higher stress

**Result: supported descriptively.** Average sleep drops steadily from the Low- to the High-stress group, and the Spearman correlation between sleep hours and stress rank is strongly negative.

In [ ]:
sleep_by_stress = df.groupby('stress_level', observed=True)['sleep_hours'].mean().round(2)
print('Average sleep hours by stress level:')
print(sleep_by_stress)

rho_sleep, p_sleep = stats.spearmanr(df['sleep_hours'], df['stress_rank'])
print(f'\nSpearman correlation (sleep hours vs. stress): rho = {rho_sleep:.3f}, p = {p_sleep:.2e}')

In [ ]:
plt.figure(figsize=(7, 5))
sleep_by_stress.plot(kind='bar', edgecolor='white')
plt.title('Average Sleep Hours by Stress Level')
plt.xlabel('Stress Level')
plt.ylabel('Average Sleep Hours')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Figure 8 - Average Sleep Hours by Stress Level

Low-stress students average about **7.97 hours** of sleep, medium-stress students about **6.76 hours**, and high-stress students about **5.01 hours** (rho approx -0.603). This is a large, monotonic gap and is the strongest relationship found in the dataset.

### Hypothesis 2: More screen time will be associated with higher stress

**Result: supported descriptively.** Average screen time rises steadily from the Low- to the High-stress group, and the Spearman correlation is strongly positive.

In [ ]:
screen_by_stress = df.groupby('stress_level', observed=True)['screen_time_hours'].mean().round(2)
print('Average screen time (hours) by stress level:')
print(screen_by_stress)

rho_screen, p_screen = stats.spearmanr(df['screen_time_hours'], df['stress_rank'])
print(f'\nSpearman correlation (screen time vs. stress): rho = {rho_screen:.3f}, p = {p_screen:.2e}')

In [ ]:
plt.figure(figsize=(7, 5))
screen_by_stress.plot(kind='bar', edgecolor='white', color='darkorange')
plt.title('Average Screen Time by Stress Level')
plt.xlabel('Stress Level')
plt.ylabel('Average Screen Time (Hours)')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Figure 9 - Average Screen Time by Stress Level

Low-stress students average about **3.33 hours** of screen time, medium-stress students about **7.09 hours**, and high-stress students about **9.39 hours** (rho approx +0.549). Interestingly, screen time shows almost no relationship with sleep (Section 5) but a clear relationship with stress directly - suggesting the two variables affect stress through separate pathways rather than screen time acting on stress purely by displacing sleep.

### Hypothesis 3: A greater amount of exercise will reduce stress

**Result: not supported.** The dataset only records physical activity as a binary Yes/No, not a duration or intensity. The proportion of students reporting high stress is nearly identical between the two groups.

In [ ]:
high_stress_rate = df.groupby('physical_activity')['stress_level'].apply(lambda s: (s == 'High').mean() * 100).round(1)
print('Percent of students reporting High stress, by physical activity:')
print(high_stress_rate)

In [ ]:
plt.figure(figsize=(6, 5))
high_stress_rate.plot(kind='bar', edgecolor='white', color='seagreen')
plt.title('High-Stress Rate by Physical Activity')
plt.xlabel('Reports Physical Activity')
plt.ylabel('Percent High Stress')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Figure 10 - High-Stress Rate by Physical Activity

21.1% of students who report **no** physical activity fall in the High-stress group, versus 19.7% of students who report physical activity. A gap that small is not a meaningful difference given a binary yes/no measure of activity, so this hypothesis is not supported by the data as collected. A duration- or frequency-based activity variable would be needed to test it properly.

## 7. Follow-Up Questions

Several follow-up questions were raised while exploring the numerical distributions (Sections 3-4). Each is answered here directly against the data.

### Do students sleeping less than 6 hours consume more caffeine?

**No.** Average caffeine intake is nearly identical for the two groups.

In [ ]:
df['sleep_lt6'] = df['sleep_hours'] < 6
caffeine_by_sleep = df.groupby('sleep_lt6')['caffeine_intake'].mean().round(2)
caffeine_by_sleep.index = caffeine_by_sleep.index.map({True: '<6 hours sleep', False: '6+ hours sleep'})
caffeine_by_sleep

### Is more screen time associated with greater academic pressure?

**No clear relationship.** Average screen time is broadly similar across Low, Medium, and High academic-pressure groups.

In [ ]:
screen_by_pressure = df.groupby('academic_pressure', observed=True)['screen_time_hours'].mean().round(2)
screen_by_pressure = screen_by_pressure.reindex(['Low', 'Medium', 'High'])
screen_by_pressure

### Does high-stress frequency change with age?

**No clear pattern.** The percentage of students reporting High stress fluctuates across ages with no consistent upward or downward trend.

In [ ]:
high_stress_by_age = df.groupby('age')['stress_level'].apply(lambda s: (s == 'High').mean() * 100).round(1)
high_stress_by_age

In [ ]:
plt.figure(figsize=(8, 5))
high_stress_by_age.plot(kind='bar', edgecolor='white', color='slateblue')
plt.title('Percent High Stress by Age')
plt.xlabel('Age')
plt.ylabel('Percent High Stress')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Do students who study more report less physical activity?

**No meaningful difference.** Average study hours are close between students who do and don't report physical activity.

In [ ]:
study_by_activity = df.groupby('physical_activity')['study_hours'].mean().round(2)
study_by_activity

### Does more caffeine correspond to more study time?

**Essentially no relationship.** The Pearson correlation between caffeine intake and study hours is close to zero.

In [ ]:
caffeine_study_corr = df['caffeine_intake'].corr(df['study_hours'])
print(f'Pearson correlation (caffeine intake vs. study hours): {caffeine_study_corr:.3f}')

### Does study time take away from sleep?

**Essentially no relationship.** The Pearson correlation between study hours and sleep hours is close to zero.

In [ ]:
study_sleep_corr = df['study_hours'].corr(df['sleep_hours'])
print(f'Pearson correlation (study hours vs. sleep hours): {study_sleep_corr:.3f}')

### Do STEM students study more than Business students?

**Cannot be answered with this dataset** - there is no `major` or field-of-study variable to group by.

### Do students who study more earn higher grades?

**Cannot be answered with this dataset** - there is no grades/GPA variable.

### Does sleep affect mental clarity or aptitude?

**Cannot be answered with this dataset** - there is no cognitive-performance or test-score variable.

## 8. Interpretation & Limitations

Several features of the dataset prompted caution in interpreting the results:

- Several numerical-variable distributions appeared unusually uniform.
- The scatterplots for screen time vs. sleep and caffeine intake vs. sleep showed little obvious structure, even though sleep and screen time each show a clear, monotonic relationship with `stress_level` directly (Section 6). This suggests sleep and screen time act on stress along largely separate pathways, rather than screen time driving stress mainly by cutting into sleep.
- The original analysis therefore raised the possibility that the dataset was synthetically generated; the clean group-level patterns in Section 6 are also consistent with either a real, well-behaved dataset or a synthetic one, so this remains an open question rather than a settled conclusion.
- The meaning of some variables is not fully specified. For example, it is unclear whether screen time and study hours are completely independent measures.
- Physical activity is recorded as a binary Yes/No, which is too coarse to properly test the exercise-stress hypothesis; a duration or frequency measure would be needed.
- Several follow-up questions (major/field of study, grades, cognitive performance) cannot be answered at all, because the dataset does not include those variables.

These limitations mean that the observed relationships should be treated as exploratory rather than causal or definitive.

## 9. Conclusion

This exploratory analysis examined distributions and relationships among several student lifestyle and academic variables, and then tested each of the three original hypotheses directly against `stress_level`.

- **Sleep -> stress:** supported. Average sleep falls from 7.97 hours (Low stress) to 6.76 hours (Medium) to 5.01 hours (High), with a Spearman rho of about -0.603 - the strongest relationship in the dataset.
- **Screen time -> stress:** supported. Average screen time rises from 3.33 hours (Low stress) to 7.09 hours (Medium) to 9.39 hours (High), with a Spearman rho of about +0.549.
- **Physical activity -> stress:** not supported. High-stress rates are nearly identical for students who do (19.7%) and don't (21.1%) report physical activity, though the binary Yes/No measure limits how much this test can show.

Of the eight follow-up questions raised during the initial exploration, two (caffeine vs. sleep duration, screen time vs. academic pressure) showed no meaningful relationship, two more (age vs. high-stress frequency, study hours vs. physical activity) showed no clear pattern, two (caffeine vs. study hours, study hours vs. sleep) showed essentially no correlation, and three (major vs. study hours, study hours vs. grades, sleep vs. cognitive performance) could not be answered because the dataset lacks the necessary variables.

Taken together, sleep and screen time both show clear, monotonic descriptive relationships with stress, while most of the other variables examined do not show meaningful relationships with each other. The analysis demonstrates a full exploratory data-analysis workflow: loading data, inspecting variables, calculating descriptive statistics, visualizing distributions, examining relationships, testing hypotheses against the target variable, answering follow-up questions, and critically evaluating limitations - while stopping short of any causal claims.